In [ ]:
#1
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
!pip install -q transformers datasets evaluate sacrebleu sentencepiece torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 11.5 MB/s eta 0:00:00


In [5]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import evaluate
import torch

In [6]:
dataset = load_dataset(
    "csv",
    data_files="final-dataset.csv"
)

dataset = dataset["train"].train_test_split(
    test_size=0.2,
    seed=42
)

Generating train split: 0 examples [00:00, ? examples/s]

In [31]:
model_name = "facebook/nllb-200-distilled-600M"

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.src_lang = "dog_Deva"

model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

model = model.to("cuda")

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

In [32]:
print(model.device)

cuda:0


In [33]:
def translate(text):

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=128
    ).to(device)

    outputs = model.generate(
        **inputs,
        forced_bos_token_id=tokenizer.convert_tokens_to_ids("eng_Latn"),
        max_length=128
    )

    return tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

In [34]:
predictions = []
references = []
dogri_sentences = []

for example in dataset["test"]:
    pred = translate(example["src_dogri"])

    predictions.append(pred)
    references.append(example["tgt_english"])
    dogri_sentences.append(example["src_dogri"])

In [35]:
pred = translate(dataset["test"][0]["src_dogri"])
print(pred)

Chloe came to the room.


In [36]:
score = meteor.compute(
    predictions=predictions,
    references=references
)

print("Baseline METEOR:", score["meteor"])

Baseline METEOR: 0.329281747403023


In [18]:
print(type(tokenizer))

<class 'transformers.models.nllb.tokenization_nllb.NllbTokenizer'>


In [19]:
print(dataset["test"].column_names)

['src_dogri', 'tgt_english']


In [37]:
model.save_pretrained("/content/drive/MyDrive/dogri3_model")
tokenizer.save_pretrained("/content/drive/MyDrive/dogri3_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/content/drive/MyDrive/dogri3_model/tokenizer_config.json',
 '/content/drive/MyDrive/dogri3_model/tokenizer.json')

In [39]:
import json

scores = {
    "meteor": score["meteor"]
}

with open("/content/drive/MyDrive/baseline_scores.json", "w") as f:
    json.dump(scores, f)